In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

#### 関数定義

In [8]:
#オシロスコープのデータを表示する関数
def plot_files(filenames):
    filenames = [f"../Datasets/{name}" for name in filenames] #パスを指定する。各自のパスを書くこと
    num_files = len(filenames)
    ncols = 2
    nrows = math.ceil(num_files / ncols)

    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows), sharex=False)
    axs = axs.flatten()  # 配列を1次元化してインデックスしやすくする

    for idx, fname in enumerate(filenames):
        try:
            with open(fname, 'r') as f:
                lines = f.readlines()

            # データ開始行
            for i, line in enumerate(lines):
                if line.strip().lower().startswith("index"):
                    data_start = i
                    break

            # 時間間隔取得
            time_line = [line for line in lines if "Time interval" in line][0]
            time_interval_str = time_line.split(":")[1].strip().split()[0].replace("uS", "")
            time_interval_s = float(time_interval_str) * 1e-6  # μs → s

            #タイトル設定
            title=input("グラフのタイトルを入力: ")

            # データ読み込み
            df = pd.read_csv(fname, sep='\t', skiprows=data_start)
            df["Time (s)"] = df["index"] * time_interval_s

            # グラフ描画（CH1とCH2を同一軸にプロット）
            ax = axs[idx]
            ax.plot(df["Time (s)"], df["CH1_Voltage(mV)"], label="CH1", color='skyblue')
            ax.plot(df["Time (s)"], df["CH2_Voltage(mV)"], label="CH2", color='gold')
            ax.set_title(f"{title}", fontsize=18)
            ax.set_xlabel("t [s]", fontsize=16)
            ax.set_ylabel("V [mV]", fontsize=16)
            ax.grid(True)
            ax.legend(loc="upper right", framealpha=1.0, fontsize=14)

        except Exception as e:
            print(f"ファイル {fname} の読み込み中にエラーが発生しました: {e}")

    # 不要な空欄の軸を非表示（ファイル数が奇数のとき）
    for j in range(len(filenames), len(axs)):
        fig.delaxes(axs[j])
        
    plt.tight_layout()
    save_name=input("保存するファイル名を入力(.pngをつけること！): ")
    plt.savefig(f"{save_name}")
    plt.show()

In [6]:
#ゲイン、位相を求める関数
def gain(v_out, v_in):
    gain = 20 * np.log10(v_out/v_in)
    return gain

def phase(t, T):
    phase = 360 * t/T
    return phase

In [7]:
#CSVファイルのボード線図の描画関数
def plot_bode(freq, gain, phase):

    # 軸ラベル・タイトル
    title = input("グラフのタイトルを入力してください: ").strip()
    
    #フォントサイズ
    label_size = 14 
    title_size = 16
    

    # 描画
    fig, ax1 = plt.subplots(figsize=(9, 5))

    ax1.set_xscale('log')
    ax1.plot(freq, gain, 'b-o', label="電圧利得 [dB]", color=skyblue)
    ax1.set_xlabel("周波数 [Hz]", fontsize=label_size)
    ax1.set_ylabel("電圧利得 [dB]", color='blue', fontsize=label_size)
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.grid(which='both', linestyle='--', linewidth=0.5)

    # 位相差のプロット
    ax2 = ax1.twinx()
    ax2.plot(freq, phase, 'r-s', label="位相差 [°]")
    ax2.set_ylabel("位相差 [°]", color='red', fontsize=label_size)
    ax2.tick_params(axis='y', labelcolor='crimson')

    # タイトルと凡例
    plt.title(title, fontsize=title_size)
    fig.tight_layout()
    plt.show()

In [10]:
#CSVデータの編集関数

def data_formatt(filename):
    #tとTを定義
    t = input("tの値を入力: ")
    T = input("Tの値を入力: ")
    df = pd.read_csv("filename")
    df["ゲイン"] = df.apply(lambda row: gain(row["v_out"], row["v_in"]), axis=1)
    df["位相"] = df.apply(lambda row: phase(t, T, axis=1))
    return df

#### 反転増幅回路

In [12]:
freq = df["周波数"]
gain = df["ゲイン"]
phase = df["位相"]

plot_bode(freq, gain, phase)

NameError: name 'df' is not defined